In [ ]:
# Cell 0: Sync training data from GitHub (auto-updates when you add clauses)
!git clone https://github.com/mail2vimal11-arch/lexara-api.git /kaggle/working/lexara-api

import sys
sys.path.insert(0, '/kaggle/working/lexara-api')
print('Repo cloned and added to path')

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers accelerate bitsandbytes peft trl datasets torch

In [ ]:
# Cell 2: Generate training data from latest repo clauses
import json

# Import directly from the cloned repo
from app.services.reference_data import ALL_REFERENCE_CLAUSES
from app.services.clause_service import CLAUSE_DB

training_data = []

# Classification tasks from FAISS clauses
for clause in ALL_REFERENCE_CLAUSES:
    training_data.append({
        'instruction': 'Classify this procurement clause. Return JSON with clause_type, risk_level, jurisdiction, and explanation.',
        'input': clause['clause_text'],
        'output': json.dumps({
            'clause_type': clause['clause_type'],
            'subtype': clause.get('subtype', ''),
            'risk_level': clause['risk_level'],
            'jurisdiction': clause['jurisdiction'],
            'source': clause['source'],
            'explanation': f"This is a {clause['clause_type'].replace('_', ' ')} clause from {clause['source']} applicable to {clause['jurisdiction']} jurisdiction."
        })
    })

# Standard clause retrieval
for clause in ALL_REFERENCE_CLAUSES:
    ctype = clause['clause_type'].replace('_', ' ')
    training_data.append({
        'instruction': f'Provide standard procurement clause language for {ctype} in {clause["jurisdiction"]} jurisdiction.',
        'input': '',
        'output': clause['clause_text']
    })

# Risk analysis for medium/high risk clauses
for clause in ALL_REFERENCE_CLAUSES:
    if clause['risk_level'] in ('Medium', 'High'):
        training_data.append({
            'instruction': 'Analyze this clause for procurement risks and suggest improvements. Return JSON.',
            'input': clause['clause_text'],
            'output': json.dumps({
                'clause_type': clause['clause_type'],
                'risk_level': clause['risk_level'],
                'risks': [f'This clause has {clause["risk_level"].lower()} risk characteristics typical of {clause["clause_type"].replace("_", " ")} provisions'],
                'recommendation': 'Review with legal counsel to ensure alignment with organizational risk tolerance.'
            })
        })

# SACC library Q&A
for clause in CLAUSE_DB:
    training_data.append({
        'instruction': f'What is the standard Canadian procurement clause for {clause["title"]}?',
        'input': '',
        'output': f'{clause["text"]}\\nSource: {clause["source"]}\\nNotes: {clause["notes"]}'
    })
    training_data.append({
        'instruction': 'Find relevant procurement clauses for: ' + ', '.join(clause['tags'][:3]),
        'input': '',
        'output': f'{clause["title"]} ({clause["category"]}):\\n{clause["text"]}'
    })

print(f'Generated {len(training_data)} training examples from latest repo')
print(f'  FAISS clauses: {len(ALL_REFERENCE_CLAUSES)}')
print(f'  SACC library: {len(CLAUSE_DB)}')

In [ ]:
# Cell 2b: Add advanced training data from Apr 2026 modules
from app.services.reference_data.training_scenarios import SCENARIO_TRAINING_DATA
from app.services.reference_data.training_clause_pairs import (
    CLAUSE_PAIR_TRAINING_DATA,
    METADATA_EXTRACTION_TRAINING_DATA,
    METADATA_EXTRACTION_TRAINING_DATA_EXTRA,
    PROCUREMENT_PROCESS_TRAINING_DATA,
)
from app.services.reference_data.training_drafting_style import ALL_DRAFTING_STYLE_TRAINING
from app.services.reference_data.training_insurance import ALL_INSURANCE_TRAINING
from app.services.reference_data.training_preprocessor import ALL_PREPROCESSOR_TRAINING

# Scenario reasoning chains (6 complex procurement disputes)
for s in SCENARIO_TRAINING_DATA:
    training_data.append({
        'instruction': s['instruction'],
        'input': s['context'],
        'output': s['response'],
    })
print(f'+ {len(SCENARIO_TRAINING_DATA)} scenario reasoning chains')

# Clause pairs: vendor-friendly vs customer-friendly (5 pairs)
for p in CLAUSE_PAIR_TRAINING_DATA:
    training_data.append(p)
print(f'+ {len(CLAUSE_PAIR_TRAINING_DATA)} clause pair comparisons')

# Metadata extraction (7 contract metadata examples)
for m in METADATA_EXTRACTION_TRAINING_DATA + METADATA_EXTRACTION_TRAINING_DATA_EXTRA:
    training_data.append(m)
print(f'+ {len(METADATA_EXTRACTION_TRAINING_DATA) + len(METADATA_EXTRACTION_TRAINING_DATA_EXTRA)} metadata extraction examples')

# Procurement process flow (1 detailed process)
for p in PROCUREMENT_PROCESS_TRAINING_DATA:
    training_data.append(p)
print(f'+ {len(PROCUREMENT_PROCESS_TRAINING_DATA)} procurement process examples')

# Legal drafting style: active voice, deontic logic, singular rule, consistency, CoT (21 examples)
for d in ALL_DRAFTING_STYLE_TRAINING:
    training_data.append(d)
print(f'+ {len(ALL_DRAFTING_STYLE_TRAINING)} drafting style examples')

# Insurance: risk mapping, thresholds, terminology, SACC, CoT, red flags (15 examples)
for i in ALL_INSURANCE_TRAINING:
    training_data.append(i)
print(f'+ {len(ALL_INSURANCE_TRAINING)} insurance training examples')

# Document preprocessing: legalese rewrites, markdown conversion, quality audit (8 examples)
for p in ALL_PREPROCESSOR_TRAINING:
    training_data.append(p)
print(f'+ {len(ALL_PREPROCESSOR_TRAINING)} preprocessor training examples')

print(f'\nTotal training examples after advanced modules: {len(training_data)}')

In [ ]:
# Cell 2c: Load CUAD + MAUD datasets from HuggingFace (Kaggle runtime only)
from app.services.reference_data.training_cuad_maud import (
    generate_cuad_training_data,
    generate_maud_training_data,
)

try:
    cuad_data = generate_cuad_training_data(max_examples=200)
    training_data.extend(cuad_data)
    print(f'+ {len(cuad_data)} CUAD contract clause examples')
except Exception as e:
    print(f'CUAD loading skipped (OK on local): {e}')

try:
    maud_data = generate_maud_training_data(max_examples=100)
    training_data.extend(maud_data)
    print(f'+ {len(maud_data)} MAUD merger agreement examples')
except Exception as e:
    print(f'MAUD loading skipped (OK on local): {e}')

print(f'\nFinal training examples: {len(training_data)}')

In [ ]:
# Cell 2d: Load LexGLUE + Pile-of-Law (Canadian decisions) corpora
#
# Broadens coverage beyond Lexara reference clauses + CUAD + MAUD with
# three additional legal corpora streamed from the Hugging Face Hub
# (no disk hit, no download progress bar):
#
#   * LexGLUE / LEDGAR                -> contract-clause classification (100 categories)
#   * LexGLUE / case_hold             -> US case-holding multiple choice
#   * Pile-of-Law / canadian_decisions -> Canadian jurisprudence (continued-pretraining style)
#
# Each source is capped so the combined run still fits a single Kaggle T4 x2
# session (~+1100 examples total). Each loader is wrapped in try/except so a
# flaky Hub source does not break the rest of the cell.

from datasets import load_dataset

# Resolve LEDGAR label names from a tiny non-streaming peek (streaming
# datasets do not always expose the ClassLabel feature directly).
try:
    _peek = load_dataset("lex_glue", "ledgar", split="train[:1]")
    LEDGAR_LABELS = _peek.features["label"].names
except Exception as e:
    LEDGAR_LABELS = None
    print(f"Could not resolve LEDGAR label names: {e}")


# --- LEDGAR: contract clause -> ProvCategory ---------------------------------
LEDGAR_CAP = 600
try:
    if LEDGAR_LABELS is None:
        raise RuntimeError("LEDGAR label names unavailable")
    ledgar = load_dataset("lex_glue", "ledgar", split="train", streaming=True)
    n = 0
    for row in ledgar:
        if n >= LEDGAR_CAP:
            break
        label = LEDGAR_LABELS[row["label"]]
        training_data.append({
            "instruction": "Classify the following contract clause into one of the LEDGAR provision categories.",
            "input": row["text"],
            "output": label,
        })
        n += 1
    print(f"+ {n} LEDGAR clause-classification examples")
except Exception as e:
    print(f"LEDGAR loading skipped: {e}")


# --- case_hold: pick the correct legal holding ------------------------------
CASEHOLD_CAP = 300
try:
    casehold = load_dataset("lex_glue", "case_hold", split="train", streaming=True)
    n = 0
    for row in casehold:
        if n >= CASEHOLD_CAP:
            break
        endings = row["endings"]
        correct_idx = row["label"]
        choices = "\n".join(f"({chr(65 + i)}) {e}" for i, e in enumerate(endings))
        training_data.append({
            "instruction": (
                "Select the legal holding that correctly completes the citing context. "
                "Respond with the letter and the holding."
            ),
            "input": f"{row['context']}\n\nCandidate holdings:\n{choices}",
            "output": f"({chr(65 + correct_idx)}) {endings[correct_idx]}",
        })
        n += 1
    print(f"+ {n} case_hold holding-selection examples")
except Exception as e:
    print(f"case_hold loading skipped: {e}")


# --- Pile-of-Law: Canadian decisions (continued-pretraining style) ----------
# No human-written labels exist for this corpus; we use a self-supervised
# completion frame so the model gains exposure to Canadian legal register
# (Tercon, Bhasin, Hunter Engineering vocabulary, prov-court formatting).
CANLAW_CAP = 200
CANLAW_PROMPT_CHARS = 600
CANLAW_TARGET_CHARS = 400
try:
    canlaw = load_dataset(
        "pile-of-law/pile-of-law",
        "canadian_decisions",
        split="train",
        streaming=True,
        trust_remote_code=True,
    )
    n = 0
    for row in canlaw:
        text = (row.get("text") or "").strip()
        if len(text) < CANLAW_PROMPT_CHARS + CANLAW_TARGET_CHARS:
            continue
        prompt = text[:CANLAW_PROMPT_CHARS]
        target = text[CANLAW_PROMPT_CHARS:CANLAW_PROMPT_CHARS + CANLAW_TARGET_CHARS]
        training_data.append({
            "instruction": "Continue the following passage from a Canadian court decision in the same legal register.",
            "input": prompt,
            "output": target,
        })
        n += 1
        if n >= CANLAW_CAP:
            break
    print(f"+ {n} Canadian-decision continuation examples")
except Exception as e:
    print(f"Pile-of-Law canadian_decisions skipped: {e}")


print(f"\nTraining examples after legal-stack expansion: {len(training_data)}")


In [ ]:
# Cell 3: Load SaulLM-7B with 4-bit quantization
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Equall/Saul-7B-Instruct-v1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False

print(f'Model loaded: {model_name}')
print(f'Parameters: {model.num_parameters():,}')

In [ ]:
# Cell 4: Format training data for fine-tuning
from datasets import Dataset

def format_prompt(example):
    if example['input']:
        text = (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Input:\n{example['input']}\n\n"
            f"### Response:\n{example['output']}"
        )
    else:
        text = (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Response:\n{example['output']}"
        )
    return {'text': text}

formatted = [format_prompt(e) for e in training_data]
dataset = Dataset.from_list(formatted)

# Split 90/10 for train/eval
split = dataset.train_test_split(test_size=0.1, seed=42)
print(f'Train: {len(split["train"])} | Eval: {len(split["test"])}')
print(f'\nSample:\n{formatted[0]["text"][:500]}')

In [ ]:
# Cell 5: Configure LoRA and train
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
    bias="none",
)

training_args = SFTConfig(
    output_dir="./lexara-legal-lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    max_seq_length=2048,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    peft_config=lora_config,
    args=training_args,
    tokenizer=tokenizer,
)

print('Starting training...')
trainer.train()
print('Training complete!')

In [ ]:
# Cell 6: Save the fine-tuned LoRA weights
trainer.save_model("./lexara-legal-lora")
tokenizer.save_pretrained("./lexara-legal-lora")
print('Model saved to ./lexara-legal-lora')

# List saved files
import os
for f in os.listdir('./lexara-legal-lora'):
    size = os.path.getsize(f'./lexara-legal-lora/{f}') / 1024 / 1024
    print(f'  {f}: {size:.1f} MB')

In [ ]:
# Cell 7: Test the fine-tuned model
from peft import PeftModel

# Reload with LoRA weights
model = PeftModel.from_pretrained(model, "./lexara-legal-lora")
model.eval()

def ask_lexara(instruction, input_text="", max_tokens=512):
    if input_text:
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    # Extract just the response part
    if '### Response:' in response:
        response = response.split('### Response:')[-1].strip()
    return response

# Test 1: Clause classification
print('=== Test 1: Clause Classification ===')
result = ask_lexara(
    'Classify this procurement clause. Return JSON with clause_type, risk_level, jurisdiction.',
    'The Contractor shall not assign this Contract without the prior written consent of Canada.'
)
print(result)

print('\n=== Test 2: CFTA Question ===')
result = ask_lexara('What are the CFTA procurement thresholds for goods and services?')
print(result)

print('\n=== Test 3: Risk Analysis ===')
result = ask_lexara(
    'Analyze this clause for procurement risks. Return JSON.',
    'The Contractor shall deliver all services as soon as possible using best efforts.'
)
print(result)

print('\n=== Test 4: Standard Clause ===')
result = ask_lexara('Provide standard indemnification clause language for Canadian federal procurement.')
print(result)

In [ ]:
# Cell 9: Upload to HuggingFace Hub
# First, login to HuggingFace (get token from https://huggingface.co/settings/tokens)
!pip install -q huggingface_hub

from huggingface_hub import notebook_login
notebook_login()  # Paste your HF token when prompted


In [ ]:
# Cell 10: Merge LoRA weights and push to HuggingFace
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Reload base model in FP16 for merging
print('Loading base model for merge...')
base_model = AutoModelForCausalLM.from_pretrained(
    'Equall/Saul-7B-Instruct-v1',
    torch_dtype=torch.float16,
    device_map='auto',
)
tokenizer = AutoTokenizer.from_pretrained('Equall/Saul-7B-Instruct-v1')

# Merge LoRA weights into base model
print('Merging LoRA weights...')
model = PeftModel.from_pretrained(base_model, './lexara-legal-lora')
model = model.merge_and_unload()

# Push to HuggingFace Hub
# CHANGE THIS to your HuggingFace username:
HF_REPO = 'mail2vimal11-arch/lexara-legal-saullm'

print(f'Pushing to {HF_REPO}...')
model.push_to_hub(HF_REPO, private=True)
tokenizer.push_to_hub(HF_REPO, private=True)

print(f'\n✅ Model uploaded to https://huggingface.co/{HF_REPO}')
print(f'\nAdd these to your VPS .env file:')
print(f'  HF_MODEL_ID={HF_REPO}')
print(f'  HF_API_TOKEN=hf_your_token_here')
print(f'  USE_LOCAL_LLM=true')
